Notebook buat menghitung skor pelafalan fonem dan membandingkannya dengan penilaian ahli. Setelah install dependency, pilih kernel Python lalu Run All. Default-nya memakai model kalibrasi terlatih dan evidence Kaldi dari satu WAV asli yang ikut repo, lalu membaca hasil evaluasi eksperimen lengkap. Ini replay evidence, bukan ekstraksi audio baru.

Kaldi dan model akustik pretrained dipakai hanya untuk forced alignment dan posterior fonem. Yang dilatih di sini kalibrator skor, bukan model ASR. Tidak ada transkripsi, streaming, atau tampilan aplikasi.

Split tetap 2.100 train, 400 validation, dan 2.500 test tanpa penutur yang bercampur. Test ini dipakai ulang dari eksperimen sebelumnya.

In [ ]:
from pathlib import Path
import os, sys, json, csv, subprocess, tempfile
import numpy as np
PROJECT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / "phoneme_scoring").is_dir())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for folder in [".cache/test-temp", ".cache/matplotlib"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MPLCONFIGDIR=str(PROJECT / ".cache/matplotlib"))
tempfile.tempdir = str(PROJECT / ".cache/test-temp")
RUN_FULL_EXTRACTION = False
RUN_TRAINING = False
RUN_LIVE_INFERENCE = False
RUN_TESTS = True
print("Folder:", PROJECT)
print("Python:", sys.executable)

Input scoring berupa audio yang sudah disiapkan: mono, 16 kHz, mengikuti preprocessing handoff. Modul ini cuma membaca dan memvalidasi PCM; tidak mengubah gain, melakukan resampling, atau memotong hening. Audio dengan format yang belum sesuai dikembalikan sebagai invalid_audio. Waktu fonem dihitung dari awal audio input ini.

Untuk ekstraksi atau training ulang, taruh wav16k di folder handoff dan siapkan runtime lewat tools/bootstrap_native_runtime.py. Aktifkan RUN_FULL_EXTRACTION dan RUN_TRAINING. RUN_LIVE_INFERENCE hanya menghitung ulang posterior dan alignment sampel, bukan live transcript.

In [ ]:
from phoneme_scoring.data import read_dataset, audit_dataset, dataset_directory
from phoneme_scoring.common import load_json, sha256_file
DATA, CACHE = dataset_directory(PROJECT), PROJECT / "cache"
datasets = read_dataset(DATA)
audit = audit_dataset(DATA, datasets)
available = sum(s["audio_files_present"] for s in audit["splits"].values())
for split, rows in datasets.items():
    print(split, len(rows), "rekaman,", len({r["speaker"] for r in rows}), "penutur")
print("WAV dataset tersedia lokal:", available, "/ 5000")
if RUN_FULL_EXTRACTION or RUN_TRAINING:
    assert available == 5000, "Ekstrak wav16k dari ZIP handoff ke folder manifest sebelum mengulang eksperimen."

Forced alignment membagi audio ke segmen fonem target. Di setiap segmen, posterior dari pdf Kaldi digabung ke 39 fonem, lalu dirata-ratakan dalam log domain:

\[
L_i(q)=\frac{1}{T_i}\sum_{t\in A_i}\log P(q\mid x_t),\qquad GOP_i=L_i(p_i)-\max_{q\in Q}L_i(q).
\]

Input kalibrator terdiri dari 39 LPP, 39 selisih terhadap target, GOP, log durasi, dan one-hot 39 fonem. Totalnya 119 fitur. GOP mentah sendiri belum sama dengan skor ahli.

Alignment mulai dari beam 10, retry 40. Kalau masih gagal, adapter mencoba beam 100 dan retry 1000 dengan audio, referensi, dan posterior yang sama. Hasil yang sudah berhasil di percobaan awal tetap dipakai. Kalau tetap nggak dapat jalur valid, responsnya gagal dengan skor null.

In [ ]:
from phoneme_scoring.backend import create_backend
CONFIG = PROJECT / "config/kaldi.native.json"
if RUN_FULL_EXTRACTION:
    subprocess.run([sys.executable, "-m", "phoneme_scoring.batch", "--data", str(DATA), "--cache", str(CACHE), "--kaldi-config", str(CONFIG), "--workers", "4", "--batch-size", "32"], check=True)
stored_metrics = load_json(PROJECT / "reports/gop/metrics.json")
print("Coverage eksperimen yang disimpan:")
for split, coverage in stored_metrics["coverage"].items():
    print(split, coverage["utterances_scored"], "/", coverage["utterances_total"], "rekaman")
assert sum(c["utterances_scored"] for c in stored_metrics["coverage"].values()) == 5000

Kalibrator dilatih untuk mendekati skor ahli. Kita bandingkan 24 kandidat dari enam model boosting dan empat nilai shrinkage ke median train.

Loss-nya absolute error atau squared error; jumlah leaf 7/15/31; minimum sampel per leaf 100/50/30; 250 iterasi; learning rate 0,08; L2 5. Seed 2026 dan early stopping dimatikan supaya prosesnya tetap. Kandidat dipilih pakai MAE validation, tanpa menambahkan bobot dari statistik seluruh dataset.

\[
y_i=\mathrm{skor\ ahli}_i/2,\quad
\hat y_i=(1-\lambda)\operatorname{clip}\left(b+\sum_{m=1}^{250}h_m(z_i),0,1\right)+\lambda\operatorname{median}(y_{train}),\quad S_i=100\hat y_i.
\]

Nilai leaf sudah memuat learning rate. Model disimpan sebagai pohon numerik di JSON. Prediksi JSON dicek sama dengan scikit-learn sebelum model disimpan. Ambang `needs_practice` dipilih dari F1 validation lalu dikunci. Label fonem salah tetap skor ahli < 0,5.

In [ ]:
from phoneme_scoring.experiments import train_experiment
from phoneme_scoring.boosting import load_calibrator
MODEL = PROJECT / "models/calibrator.json"
if RUN_TRAINING:
    assert all((CACHE / f"index_{s}.json").is_file() for s in ["train", "val", "test"]), "Aktifkan RUN_FULL_EXTRACTION untuk membuat cache eksperimen."
    train_experiment(DATA, CACHE, PROJECT / "reports/gop", MODEL)
    subprocess.run([sys.executable, "tools/plot_real_metrics.py"], check=True)
model = load_calibrator(MODEL)
metrics = load_json(PROJECT / "reports/gop/metrics.json")
assert sha256_file(MODEL) == metrics["model_sha256"]
print("Pilihan:", model.artifact["fit_parameters"])
print("Shrinkage:", model.artifact["shrink_to_median"])
print("Ambang latihan (0–1):", model.threshold)
for c in sorted(model.artifact["candidate_validation_results"], key=lambda x:x["validation_mae_raw_0_2"])[:5]:
    print(c)

In [ ]:
for split in ["val", "test"]:
    print("\n", split)
    for name, result in metrics["results"][split].items():
        p = result["phoneme"]; d = p["mispronunciation"]
        print(f"{name:18} N={p['n']:5d} MAE={p['mae_raw_0_2']:.6f} RMSE={p['rmse_raw_0_2']:.6f} Pearson={p['pearson']} F1={d['f1']:.4f} AP={d['pr_auc_average_precision']:.4f}")
print("MAE test melewati baseline:", metrics["beats_baseline_test_mae"])
# Perbandingan baseline dilaporkan apa adanya, bukan syarat notebook berhasil.
print("Coverage:", {k:(v["utterances_scored"],v["phones_scored"]) for k,v in metrics["coverage"].items()})

MAE kecil perlu dibaca bareng F1, AP, dan galat pada fonem yang skornya rendah. Mayoritas fonem di data ini mendapat nilai 2, jadi menebak 2 terus memang baseline yang kuat. Korelasi baseline konstan ditulis null karena nggak terdefinisi.

Rata-rata skor fonem per kata/kalimat cuma ringkasan ketepatan fonem. Kita tetap bandingkan dengan label accuracy kata/kalimat, tapi jangan langsung menganggap keduanya skor holistik yang sama. Fluency, intonasi, dan stress belum dinilai.

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(PROJECT / "reports/gop/test_gop_calibrated.png")))
display(Image(filename=str(PROJECT / "reports/gop/test_detection.png")))
r = metrics["results"]["test"]["gop_calibrated"]
print("Kata:", r["word_summary_vs_expert"])
print("Kalimat:", r["sentence_summary_vs_expert"])
print("Fonem skor rendah:", r["low_score_subset"])
for group, value in r["groups"]["age_group"].items():
    print(group, "MAE", value["mae_raw_0_2"], "N", value["n"])

Empat rekaman yang sebelumnya gagal sudah mendapat jalur alignment lewat retry. Ringkasan hasil yang disimpan menunjukkan prediksi masih terlalu tinggi dibanding label ahli. Hasil ini perlu ditinjau; keberhasilan alignment belum menyelesaikan salah penilaiannya.

In [ ]:
recovery = load_json(PROJECT / "reports/gop/recovery_summary.json")
if recovery["model_sha256"] == sha256_file(MODEL):
    assert recovery["extractor_id"] == model.extractor_id
else:
    print("Analisis empat rekaman di bawah berasal dari eksperimen tersimpan, bukan model yang baru dilatih.")
for row in recovery["recordings"]:
    print(row["utt_id"], row["phones"], "fonem; prediksi", round(row["predicted_mean_score_100"], 2), "vs ahli", round(row["expert_mean_score_100"], 2))
assert len(recovery["recordings"]) == 4
assert all(r["live_matches_cache"] for r in recovery["recordings"])

In [ ]:
print("Mode:", "inferensi Kaldi langsung" if RUN_LIVE_INFERENCE else "replay evidence Kaldi dari sampel WAV asli")
from phoneme_scoring import PronunciationScorer
from phoneme_scoring.evidence import CachedBackend
words = load_json(PROJECT / "examples/reference.json")["reference_words"]
backend = create_backend(CONFIG) if RUN_LIVE_INFERENCE else CachedBackend(PROJECT / "examples/evidence", model.extractor_id)
scorer = PronunciationScorer(backend, model)
result = scorer.score_pronunciation(PROJECT / "examples/real_sample.wav", None, words)
assert result["status"] == "ok", result
(PROJECT / "reports/notebook_live_result.json").write_text(json.dumps(result, indent=2)+"\n", encoding="utf-8")
for word in result["words"]:
    print(word["word"], round(word["phoneme_accuracy_summary"], 2), [(p["target"], round(p["score"], 1), p["needs_practice"]) for p in word["phones"]])
failed = scorer.score_pronunciation(np.zeros(16000), 16000, words)
assert failed["error"]["code"] == "no_speech" and failed["phoneme_accuracy_summary"] is None
print("Rekaman hening:", failed["error"])

In [ ]:
import jsonschema
schema = load_json(PROJECT / "schemas/result.schema.json")
jsonschema.validate(result, schema)
jsonschema.validate(failed, schema)
if RUN_TESTS:
    test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
    print("Log tes memakai fixture sintetis; hasilnya terpisah dari evaluasi audio asli.")
    print(test_run.stdout)
    print(test_run.stderr)
    test_run.check_returncode()
print("Kontrak output valid.")

Metrik dan grafik eksperimen tersimpan di reports/gop. CSV prediksi dibuat kalau training dijalankan ulang; file itu tidak ikut repo. Kalibrator aktif ada di models/calibrator.json. Notebook ini mencakup metode, rumus, skor per fonem, ringkasan kata/kalimat, dan evaluasi scoring terhadap ahli.

GOP mengikuti perhitungan posterior pada implementasi Kaldi (https://kaldi-asr.org/doc/compute-gop_8cc.html). Dataset dan penilaian ahli dijelaskan di paper SpeechOcean762 (https://www.isca-archive.org/interspeech_2021/zhang21x_interspeech.pdf).